In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import layers, models, callbacks

In [2]:
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [3]:
VOCAB_SIZE = 20_000
MAX_LEN = 50
EMB_DIM = 128
LSTM_UNITS = 64
BATCH_SIZE = 256
EPOCHS = 15

In [4]:
df = pd.read_csv("../data/sentiment_data.csv", usecols=["Comment", "Sentiment"])
df = df.dropna(subset=["Comment"])
df["Comment"] = df["Comment"].astype(str).str.strip()
df = df[df["Comment"] != ""]

In [5]:
conflicting = df.groupby("Comment")["Sentiment"].nunique()
conflicting = conflicting[conflicting > 1].index
print(f"Çelişkili etiketli metin sayısı: {len(conflicting)}")
df = df[~df["Comment"].isin(conflicting)]

Çelişkili etiketli metin sayısı: 166


In [6]:
print(f"Tekrar sayısı (silinmeden önce): {df['Comment'].duplicated().sum()}")
df = df.drop_duplicates(subset="Comment", keep="first")
print(f"Temizlenmiş veri boyutu: {len(df)}")

Tekrar sayısı (silinmeden önce): 25575
Temizlenmiş veri boyutu: 213991


In [7]:
X_text = df["Comment"].values
y = df["Sentiment"].values

In [8]:
X_train_txt, X_temp, y_train, y_temp = train_test_split(
    X_text, y, test_size=0.30, stratify=y, random_state=SEED
)
X_val_txt, X_test_txt, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED
)
print(f"Train: {len(X_train_txt)}  Val: {len(X_val_txt)}  Test: {len(X_test_txt)}")

Train: 149793  Val: 32099  Test: 32099


In [9]:
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_txt)

In [10]:
def to_sequences(texts):
    seqs = tokenizer.texts_to_sequences(texts)
    return pad_sequences(seqs, maxlen=MAX_LEN, padding="pre", truncating="post")

In [11]:
X_train = to_sequences(X_train_txt)
X_val = to_sequences(X_val_txt)
X_test = to_sequences(X_test_txt)

In [12]:
classes = np.unique(y_train)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weight = dict(zip(classes.tolist(), weights.tolist()))
print("Sınıf ağırlıkları:", class_weight)

Sınıf ağırlıkları: {0: 1.4788673992240027, 1: 0.9958912579532082, 2: 0.7577587907668493}


In [13]:
num_classes = len(classes)

In [14]:
model = models.Sequential([
    layers.Input(shape=(MAX_LEN,)),
    layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMB_DIM, mask_zero=True),
    layers.Bidirectional(layers.LSTM(LSTM_UNITS)),
    layers.Dropout(0.3),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation="softmax"),
])

In [15]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ (None, 50, 128)             │       2,560,000 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ bidirectional (Bidirectional)        │ (None, 128)                 │          98,816 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 64)                  │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 3)                   │             195 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,667,267 (10.17 MB)

 Trainable params: 2,667,267 (10.17 MB)

 Non-trainable params: 0 (0.00 B)

In [16]:
cbs = [
    callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=1, min_lr=1e-5),
]

In [17]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weight,
    callbacks=cbs,
    verbose=1,
)

Epoch 1/15
586/586 ━━━━━━━━━━━━━━━━━━━━ 216s 345ms/step - accuracy: 0.7550 - loss: 0.6133 - val_accuracy: 0.8302 - val_loss: 0.4678 - learning_rate: 0.0010
Epoch 2/15
586/586 ━━━━━━━━━━━━━━━━━━━━ 205s 349ms/step - accuracy: 0.8489 - loss: 0.4153 - val_accuracy: 0.8328 - val_loss: 0.4674 - learning_rate: 0.0010
Epoch 3/15
586/586 ━━━━━━━━━━━━━━━━━━━━ 197s 337ms/step - accuracy: 0.8671 - loss: 0.3558 - val_accuracy: 0.8261 - val_loss: 0.5066 - learning_rate: 0.0010
Epoch 4/15
586/586 ━━━━━━━━━━━━━━━━━━━━ 196s 326ms/step - accuracy: 0.8937 - loss: 0.2845 - val_accuracy: 0.8244 - val_loss: 0.5551 - learning_rate: 5.0000e-04
Epoch 5/15
586/586 ━━━━━━━━━━━━━━━━━━━━ 208s 337ms/step - accuracy: 0.9139 - loss: 0.2302 - val_accuracy: 0.8203 - val_loss: 0.6303 - learning_rate: 2.5000e-04


In [18]:
probs = model.predict(X_test, batch_size=BATCH_SIZE)
y_pred = probs.argmax(axis=1)

126/126 ━━━━━━━━━━━━━━━━━━━━ 17s 128ms/step


In [19]:
majority_acc = (y_test == pd.Series(y_test).mode()[0]).mean()
print(f"\nBaseline (hep çoğunluk sınıfı) accuracy: {majority_acc:.4f}")
print(f"Test accuracy: {(y_pred == y_test).mean():.4f}")
print(f"Test macro F1: {f1_score(y_test, y_pred, average='macro'):.4f}")
print("\nSınıflandırma raporu:")
print(classification_report(y_test, y_pred, digits=4))
print("Confusion matrix (satır=gerçek, sütun=tahmin):")
print(confusion_matrix(y_test, y_pred))


Baseline (hep çoğunluk sınıfı) accuracy: 0.4399
Test accuracy: 0.8303
Test macro F1: 0.8215

Sınıflandırma raporu:
              precision    recall  f1-score   support

           0     0.7298    0.8169    0.7709      7235
           1     0.8193    0.8357    0.8274     10744
           2     0.9019    0.8331    0.8661     14120

    accuracy                         0.8303     32099
   macro avg     0.8170    0.8286    0.8215     32099
weighted avg     0.8355    0.8303    0.8317     32099

Confusion matrix (satır=gerçek, sütun=tahmin):
[[ 5910   788   537]
 [ 1023  8979   742]
 [ 1165  1192 11763]]


In [20]:
model.save("../model/sentiment_lstm.keras")
tokenizer_json = tokenizer.to_json()
with open("tokenizer.json", "w", encoding="utf-8") as f:
    f.write(tokenizer_json)

In [21]:
def predict_sentiment(texts):
    seqs = to_sequences(texts)
    p = model.predict(seqs, verbose=0)
    return p.argmax(axis=1), p

In [22]:
if __name__ == "__main__":
    samples = [
        "apple pay is so convenient and secure",
        "the store refused to take apple pay again, very annoying",
    ]
    labels, probs_new = predict_sentiment(samples)
    for t, l, p in zip(samples, labels, probs_new):
        print(f"{t!r} -> sınıf {l}  olasılıklar {np.round(p, 3)}")

'apple pay is so convenient and secure' -> sınıf 2  olasılıklar [0.135 0.272 0.593]
'the store refused to take apple pay again, very annoying' -> sınıf 0  olasılıklar [0.81  0.165 0.025]
